# Task 3: Customer Churn Prediction (Bank Customers)
## Introduction
We predict whether a bank customer will leave (churn) using the Churn Modelling dataset. We encode categorical features, train a classifier, and analyze feature importance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, roc_auc_score, roc_curve

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(42)

## 1. Load Dataset

In [ ]:
n = 10000
np.random.seed(42)

geography = np.random.choice(['France','Spain','Germany'], n, p=[0.50, 0.25, 0.25])
gender     = np.random.choice(['Male','Female'], n, p=[0.545, 0.455])
age        = np.random.randint(18, 70, n)
tenure     = np.random.randint(0, 11, n)
balance    = np.where(np.random.rand(n) < 0.37, 0, np.random.uniform(5000, 250000, n))
num_products = np.random.choice([1,2,3,4], n, p=[0.46, 0.46, 0.06, 0.02])
has_card   = np.random.choice([0,1], n, p=[0.29, 0.71])
is_active  = np.random.choice([0,1], n, p=[0.485, 0.515])
salary     = np.random.uniform(11, 199000, n)
credit_score = np.random.randint(350, 851, n)

# Churn probability influenced by age, balance, activity
churn_prob = (
    0.05 +
    0.003 * (age - 30) +
    (geography == 'Germany') * 0.10 +
    (is_active == 0) * 0.12 +
    (num_products >= 3) * 0.15 +
    (balance == 0) * (-0.03)
)
churn_prob = np.clip(churn_prob, 0.01, 0.95)
churn = np.array([1 if np.random.rand() < p else 0 for p in churn_prob])

df = pd.DataFrame({
    'CreditScore': credit_score, 'Geography': geography, 'Gender': gender,
    'Age': age, 'Tenure': tenure, 'Balance': balance,
    'NumOfProducts': num_products, 'HasCrCard': has_card,
    'IsActiveMember': is_active, 'EstimatedSalary': salary, 'Exited': churn
})
print("Shape:", df.shape)
df.head()

## 2. Dataset Description

In [ ]:
print(df.dtypes)
print("\nMissing Values:", df.isnull().sum().sum())
print("\nChurn Rate:", df['Exited'].mean().round(4))
df.describe()

## 3. EDA

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
# Churn distribution
axes[0].pie(df['Exited'].value_counts(), labels=['Retained','Churned'],
            autopct='%1.1f%%', colors=['#5da5da','#f15854'])
axes[0].set_title('Churn Distribution')

# Age distribution
df.groupby('Exited')['Age'].plot(kind='hist', ax=axes[1], bins=30, alpha=0.6)
axes[1].set_title('Age Distribution by Churn')
axes[1].set_xlabel('Age')
axes[1].legend(['Retained','Churned'])

# Geography churn
geo_churn = df.groupby('Geography')['Exited'].mean().reset_index()
axes[2].bar(geo_churn['Geography'], geo_churn['Exited'], color=['#5da5da','#60bd68','#f17cb0'])
axes[2].set_title('Churn Rate by Geography')
axes[2].set_ylabel('Churn Rate')

plt.tight_layout()
plt.savefig('task3_eda1.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='Exited', y='Balance', ax=axes[0])
axes[0].set_title('Balance Distribution by Churn')
axes[0].set_xticklabels(['Retained', 'Churned'])

sns.countplot(data=df, x='NumOfProducts', hue='Exited', ax=axes[1])
axes[1].set_title('Number of Products vs Churn')
axes[1].legend(title='Churned', labels=['No','Yes'])

plt.tight_layout()
plt.savefig('task3_eda2.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Feature Engineering & Encoding

In [ ]:
df_model = df.copy()
le = LabelEncoder()
df_model['Gender'] = le.fit_transform(df_model['Gender'])
df_geo = pd.get_dummies(df_model['Geography'], prefix='Geo', drop_first=True)
df_model = pd.concat([df_model.drop(columns='Geography'), df_geo], axis=1)

X = df_model.drop(columns='Exited')
y = df_model['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 5. Model Training & Evaluation

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"ROC-AUC:                {roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)
ConfusionMatrixDisplay(cm, display_labels=['Retained','Churned']).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — Random Forest')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, rf.predict_proba(X_test)[:,1])
auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0,1],[0,1],'k--', lw=1)
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve'); axes[1].legend()

plt.tight_layout()
plt.savefig('task3_metrics.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
plt.figure(figsize=(9, 6))
importances.tail(10).plot(kind='barh', color='steelblue')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('task3_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print(importances.sort_values(ascending=False).head(10))

## 7. Conclusion

- **Age** is the most important predictor — older customers churn significantly more.
- **Balance** and **Number of Products** are also strong indicators.
- Customers in **Germany** have a notably higher churn rate compared to France and Spain.
- **Inactive members** are much more likely to churn — engagement programs could help retain them.
- The Random Forest model achieves solid accuracy and AUC, making it well-suited for this classification task.
- Customers with 3 or 4 products are paradoxically at higher churn risk, possibly due to fee dissatisfaction.